# 03c. Timeseries Forecasting — LSTM, BiGRU, CNN-LSTM

**Authors:** Fajar Laksono
**Methodology:** CRISP-ML(Q) + CAMS DevOps
**Last Updated:** 2026-05-12

This notebook loads CPU readings via DuckDB out-of-core and trains timeseries models per-VM.

---

In [ ]:
import os, sys, warnings, pathlib
import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')
warnings.filterwarnings('ignore')
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
import os
from pathlib import Path
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
DATA_DIR = Path(os.getenv('DATA_DIR', 'data/transformed/parquet'))
if not DATA_DIR.is_absolute():
    DATA_DIR = (PROJECT_ROOT / DATA_DIR).resolve()
sys.path.insert(0, str(PROJECT_ROOT))

# Thin imports
from app.src.features import create_sequences, load_cpu_readings

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

# Statistical
from statsmodels.tsa.arima.model import ARIMA

# Deep learning
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, GRU, Bidirectional, Dense, Dropout, Conv1D, MaxPooling1D
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    from tensorflow.keras.optimizers import Adam
    TF_AVAILABLE = True
    print('TensorFlow available.')
except ImportError:
    TF_AVAILABLE = False
    print('TensorFlow not installed. Only ARIMA baseline will run.')
    print('To install: pip install tensorflow')

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
if TF_AVAILABLE:
    tf.random.set_seed(RANDOM_STATE)

# DATA_DIR now from .env (see first cell)
print('All libraries imported successfully.')


## 8. Deep Learning — Timeseries Forecasting

**CRISP-ML(Q) Phase:** Modeling / Evaluation

**Literature basis:** LSTM, BiGRU, CNN-LSTM recommended for forecasting.

**Note:** This section requires TensorFlow. If unavailable, only ARIMA baseline will run.

### 8.1. Load CPU Readings Timeseries

**Business Question:** Can deep learning models predict future CPU utilization from historical timeseries?

In [ ]:
# Load CPU readings via DuckDB glob (out-of-core, no full materialization)
con = duckdb.connect(':memory:')
cpu_shards = sorted(DATA_DIR.glob('cpu_readings/*.parquet'))

if cpu_shards:
    # DuckDB glob GROUP BY — no full load into memory
    cpu_vm_stats = con.execute(f"""
        SELECT vm_id,
               COUNT(*) AS count,
               MIN(timestamp) AS min_ts,
               MAX(timestamp) AS max_ts,
               (MAX(timestamp) - MIN(timestamp)) / 3600.0 AS duration_hours
        FROM read_parquet('{DATA_DIR}/cpu_readings/*.parquet')
        GROUP BY vm_id
    """).fetchdf()

    top_n = min(5, len(cpu_vm_stats))
    top_vms = cpu_vm_stats.nlargest(top_n, 'duration_hours')
    print(f'\u2713 Discovered {len(cpu_vm_stats)} VMs in {len(cpu_shards)} shards')
    print(f'  Selected {len(top_vms)} VMs with longest traces:')
    for vm_id in top_vms['vm_id']:
        row = top_vms[top_vms['vm_id'] == vm_id].iloc[0]
        print(f'    {vm_id}: {int(row["count"]):,} readings, {row["duration_hours"]:.1f}h')

    # Load only the selected VMs' data (DuckDB filters at storage level)
    vm_ids_quoted = [f"'{v}'" for v in top_vms['vm_id'].tolist()]
    cpu_traces = con.execute(f"""
        SELECT vm_id, timestamp, avg_cpu
        FROM read_parquet('{DATA_DIR}/cpu_readings/*.parquet')
        WHERE vm_id IN ({', '.join(vm_ids_quoted)})
        ORDER BY vm_id, timestamp
    """).fetchdf()

    vm_series_dict = {}
    for vm_id in top_vms['vm_id']:
        series = cpu_traces[cpu_traces['vm_id'] == vm_id].reset_index(drop=True)
        vm_series_dict[vm_id] = series['avg_cpu'].values

    mem_mb = cpu_traces.memory_usage(deep=True).sum() / 1024**2
    print(f'  Total loaded: {len(cpu_traces):,} rows ({mem_mb:.1f} MB)')
else:
    print('No CPU readings available. Using synthetic data for demonstration.')
    np.random.seed(RANDOM_STATE)
    t = np.arange(1000)
    synthetic = 50 + 20 * np.sin(2*np.pi*t/24) + 10 * np.sin(2*np.pi*t/(24*7)) + np.random.normal(0, 5, 1000)
    vm_series_dict = {'synthetic': synthetic}
    print(f'  Created synthetic timeseries: {len(synthetic)} steps')


### 8.2. Data Preparation

In [ ]:
LOOKBACK = 24
FORECAST_HORIZON = 1

from sklearn.preprocessing import MinMaxScaler

# Prepare sequences for each VM
all_seq = {}  # vm_id -> {X_train, X_val, X_test, y_train, y_val, y_test, scaler}
for vm_id, values in vm_series_dict.items():
    # Normalize
    ts_scaler = MinMaxScaler()
    data_scaled = ts_scaler.fit_transform(values.reshape(-1, 1)).flatten()

    # Create sequences
    X_seq, y_seq = create_sequences(data_scaled, lookback=LOOKBACK, forecast_horizon=FORECAST_HORIZON)

    # Chronological split
    n = len(X_seq)
    n_train = int(n * 0.7)
    n_val = int(n * 0.15)

    all_seq[vm_id] = {
        'X_train': X_seq[:n_train],
        'y_train': y_seq[:n_train],
        'X_val': X_seq[n_train:n_train + n_val],
        'y_val': y_seq[n_train:n_train + n_val],
        'X_test': X_seq[n_train + n_val:],
        'y_test': y_seq[n_train + n_val:],
        'scaler': ts_scaler,
    }

    print(f'VM {vm_id}: {len(X_seq)} sequences '
          f'(train={n_train}, val={n_val}, test={n - n_train - n_val})')


### 8.3. ARIMA Baseline

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

# ARIMA on the first VM (raw, non-normalized)
first_vm = list(vm_series_dict.keys())[0]
first_values = vm_series_dict[first_vm]
n_first = len(first_values)

train_raw = first_values[:int(n_first * 0.7)]
test_raw = first_values[int(n_first * 0.7):]

try:
    arima = ARIMA(train_raw, order=(5, 1, 0))
    arima_fit = arima.fit()
    arima_pred = arima_fit.forecast(steps=len(test_raw))

    min_len = min(len(arima_pred), len(test_raw))
    arima_mae = mean_absolute_error(test_raw[:min_len], arima_pred[:min_len])
    arima_rmse = np.sqrt(mean_squared_error(test_raw[:min_len], arima_pred[:min_len]))
    print(f'ARIMA(5,1,0) on {first_vm}:')
    print(f'  MAE: {arima_mae:.4f}')
    print(f'  RMSE: {arima_rmse:.4f}')
except Exception as e:
    print(f'ARIMA failed: {e}')
    arima_mae = np.nan
    arima_rmse = np.nan


### 8.4. LSTM Model

**Business Question:** Can LSTM capture long-term dependencies in CPU utilization?

In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, GRU, Bidirectional, Dense, Dropout, Conv1D, MaxPooling1D
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    from tensorflow.keras.optimizers import Adam
    TF_AVAILABLE = True
    tf.random.set_seed(RANDOM_STATE)
    print('TensorFlow available. Training deep learning models across {} VMs...'.format(len(all_seq)))
except ImportError:
    print('TensorFlow not installed. Skipping deep learning models.')
    print('To run these sections: pip install tensorflow')
    TF_AVAILABLE = False

ts_results = {}  # model_name -> list of metric dicts (one per VM)

if TF_AVAILABLE:
    np.random.seed(RANDOM_STATE)

    for vm_id, data in all_seq.items():
        print(f'\n--- LSTM: {vm_id} ---')
        lstm_model = Sequential([
            LSTM(64, input_shape=(LOOKBACK, 1)),
            Dropout(0.2),
            Dense(1)
        ])
        lstm_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

        early_stop = EarlyStopping(patience=10, restore_best_weights=True)
        reduce_lr = ReduceLROnPlateau(patience=5, factor=0.5)

        lstm_model.fit(
            data['X_train'], data['y_train'],
            validation_data=(data['X_val'], data['y_val']),
            epochs=50, batch_size=32,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )

        lstm_pred = lstm_model.predict(data['X_test'], verbose=0)
        lstm_pred_raw = data['scaler'].inverse_transform(lstm_pred)
        y_test_raw = data['scaler'].inverse_transform(data['y_test'].reshape(-1, 1))

        metric = {
            'mae': mean_absolute_error(y_test_raw, lstm_pred_raw),
            'rmse': np.sqrt(mean_squared_error(y_test_raw, lstm_pred_raw)),
        }
        if 'LSTM' not in ts_results:
            ts_results['LSTM'] = []
        ts_results['LSTM'].append(metric)
        print(f'  MAE: {metric["mae"]:.4f}, RMSE: {metric["rmse"]:.4f}')



### 8.5. GRU / BiGRU Model

**Business Question:** Does BiGRU offer better performance or faster convergence than LSTM?

In [ ]:
if TF_AVAILABLE:
    for vm_id, data in all_seq.items():
        print(f'\n--- BiGRU: {vm_id} ---')
        bigru_model = Sequential([
            Bidirectional(GRU(64), input_shape=(LOOKBACK, 1)),
            Dropout(0.2),
            Dense(1)
        ])
        bigru_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

        bigru_model.fit(
            data['X_train'], data['y_train'],
            validation_data=(data['X_val'], data['y_val']),
            epochs=50, batch_size=32,
            callbacks=[EarlyStopping(patience=10, restore_best_weights=True), reduce_lr],
            verbose=0
        )

        bigru_pred_raw = data['scaler'].inverse_transform(bigru_model.predict(data['X_test'], verbose=0))
        y_test_raw = data['scaler'].inverse_transform(data['y_test'].reshape(-1, 1))

        metric = {
            'mae': mean_absolute_error(y_test_raw, bigru_pred_raw),
            'rmse': np.sqrt(mean_squared_error(y_test_raw, bigru_pred_raw)),
        }
        if 'BiGRU' not in ts_results:
            ts_results['BiGRU'] = []
        ts_results['BiGRU'].append(metric)
        print(f'  MAE: {metric["mae"]:.4f}, RMSE: {metric["rmse"]:.4f}')



### 8.6. CNN-LSTM Hybrid

**Business Question:** Can a CNN-LSTM capture both local patterns and long-term dependencies?

In [ ]:
if TF_AVAILABLE:
    for vm_id, data in all_seq.items():
        print(f'\n--- CNN-LSTM: {vm_id} ---')
        cnn_lstm = Sequential([
            Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=(LOOKBACK, 1)),
            MaxPooling1D(pool_size=2),
            LSTM(32),
            Dropout(0.1),
            Dense(1)
        ])
        cnn_lstm.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

        cnn_lstm.fit(
            data['X_train'], data['y_train'],
            validation_data=(data['X_val'], data['y_val']),
            epochs=50, batch_size=32,
            callbacks=[EarlyStopping(patience=10, restore_best_weights=True), reduce_lr],
            verbose=0
        )

        cnn_pred_raw = data['scaler'].inverse_transform(cnn_lstm.predict(data['X_test'], verbose=0))
        y_test_raw = data['scaler'].inverse_transform(data['y_test'].reshape(-1, 1))

        metric = {
            'mae': mean_absolute_error(y_test_raw, cnn_pred_raw),
            'rmse': np.sqrt(mean_squared_error(y_test_raw, cnn_pred_raw)),
        }
        if 'CNN-LSTM' not in ts_results:
            ts_results['CNN-LSTM'] = []
        ts_results['CNN-LSTM'].append(metric)
        print(f'  MAE: {metric["mae"]:.4f}, RMSE: {metric["rmse"]:.4f}')



### 8.7. Temporal Fusion Transformer Discussion

**Business Question:** Is TFT suitable for this use case?

> TFT (Temporal Fusion Transformer) by Lim et al. is SOTA for multi-horizon forecasting with interpretable attention mechanisms. It requires:
> - Multi-VM panel data (not single-series)
> - Significant computational resources
> - Complex hyperparameter tuning
> 
> **Decision:** Acknowledge TFT as recommended future work. Not implemented here due to data limitations (25/195 CPU shards available) and complexity.

Reference: Lim, B. et al. "Temporal Fusion Transformers for interpretable multi-horizon time series forecasting." International Journal of Forecasting, 2021.

### 8.8. Model Comparison & Save

In [ ]:
# ---------------------------------------------------------------------------
# TIMESERIES ACCEPTANCE GATE — CRISP-ML(Q) Quality Assurance
# ---------------------------------------------------------------------------
if ts_results:
    print("=" * 60)
    print("TIMESERIES ACCEPTANCE GATE")
    print("=" * 60)
    all_pass = True
    for model_name, metrics in ts_results.items():
        mae = metrics.get('mae', 999)
        print(f"  {'[OK]' if mae < 5 else '✗'} {model_name}: MAE = {mae:.3f}")
        if mae >= 5:
            all_pass = False
    assert all_pass, "FAIL: Timeseries models exceed MAE threshold"
    print("[OK] All timeseries models pass acceptance gate")

In [ ]:
# ---------------------------------------------------------------------------
# TIMESERIES ACCEPTANCE GATE — CRISP-ML(Q) Quality Assurance
# ---------------------------------------------------------------------------
if ts_results:
    print("=" * 60)
    print("TIMESERIES ACCEPTANCE GATE")
    print("=" * 60)
    all_pass = True
    for model_name, metrics in ts_results.items():
        mae = metrics.get('mae', 999)
        print(f"  {'[OK]' if mae < 5 else '✗'} {model_name}: MAE = {mae:.3f}")
        if mae >= 5:
            all_pass = False
    assert all_pass, "FAIL: Timeseries models exceed MAE threshold"
    print("[OK] All timeseries models pass acceptance gate")